# Ask Sage Connection Test

Use this notebook to confirm that `credentials.json` can initialize Ask Sage, list available models, send one short free-text query, and run the same structured-output classification path used by the ontology notebook. The API key is never printed.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from programming_ontology.asksage_classifier import (
    AskSageChallengeClassifier,
    build_asksage_client,
    build_asksage_openai_client,
    default_credentials_path,
    load_credentials,
    parse_asksage_credentials,
    resolve_asksage_model,
)

def mask_email(value):
    if not value or "@" not in value:
        return "configured"
    name, domain = value.split("@", 1)
    if len(name) <= 2:
        return f"{name[:1]}***@{domain}"
    return f"{name[:1]}***{name[-1:]}@{domain}"

## Load Credentials

This reads `credentials.json` from the project root. It prints only non-secret details.

In [2]:
credentials_path = default_credentials_path()
credentials_data = load_credentials(credentials_path)
credentials = parse_asksage_credentials(credentials_data, source=str(credentials_path))
selected_model = resolve_asksage_model(credentials_path=credentials_path)

print(f"Credentials file: {credentials_path}")
print(f"Ask Sage user: {mask_email(credentials.email)}")
print(f"User API URL: {credentials.user_base_url}")
print(f"Server API URL: {credentials.server_base_url}")
print(f"Selected model: {selected_model}")

Credentials file: C:\Users\Cameron.Webster\Python\ProgrammingOntology\credentials.json
Ask Sage user: c***r@nnsa.doe.gov
User API URL: https://api.nnsa.asksage.ai/user
Server API URL: https://api.nnsa.asksage.ai/server
Selected model: gpt-5.1


## Create Classic Client

In [3]:
client = build_asksage_client(credentials_path=credentials_path)
print("Ask Sage client initialized successfully.")

Ask Sage client initialized successfully.


## Check Available Models

In [4]:
models_response = client.get_models()
display(models_response)

models = models_response.get("response", models_response) if isinstance(models_response, dict) else models_response
if isinstance(models, list):
    rows = []
    for item in models:
        rows.append(item if isinstance(item, dict) else {"model": str(item)})
    display(pd.DataFrame(rows))
else:
    print("Model response was not a list; raw response is displayed above.")

{'data': [{'created': 'na',
   'id': 'aws-bedrock-claude-45-sonnet-gov',
   'name': 'aws-bedrock-claude-45-sonnet-gov',
   'object': 'model',
   'owned_by': 'organization-owner'},
  {'created': 'na',
   'id': 'aws-bedrock-nova-lite-gov',
   'name': 'aws-bedrock-nova-lite-gov',
   'object': 'model',
   'owned_by': 'organization-owner'},
  {'created': 'na',
   'id': 'aws-bedrock-nova-pro-gov',
   'name': 'aws-bedrock-nova-pro-gov',
   'object': 'model',
   'owned_by': 'organization-owner'},
  {'created': 'na',
   'id': 'aws-bedrock-nova-micro-gov',
   'name': 'aws-bedrock-nova-micro-gov',
   'object': 'model',
   'owned_by': 'organization-owner'},
  {'created': 'na',
   'id': 'aws-bedrock-titan',
   'name': 'aws-bedrock-titan',
   'object': 'model',
   'owned_by': 'organization-owner'},
  {'created': 'na',
   'id': 'llma3',
   'name': 'llma3',
   'object': 'model',
   'owned_by': 'organization-owner'},
  {'created': 'na',
   'id': 'aws-bedrock-nemotron-12b-vl-gov',
   'name': 'aws-bedroc

,model
0,aws-bedrock-claude-45-sonnet-gov
1,aws-bedrock-nova-lite-gov
2,aws-bedrock-nova-pro-gov
3,aws-bedrock-nova-micro-gov
4,aws-bedrock-titan
...,...
66,flux-2-pro
67,grok-4-20-reasoning
68,grok-4-20-non-reasoning
69,grok-4-1-fast-reasoning


## Send Free-Text Test Query

In [5]:
TEST_QUERY = "Reply with one short sentence confirming the Ask Sage connection works."

test_response = client.query(
    message=TEST_QUERY,
    persona="default",
    dataset="none",
    limit_references=0,
    temperature=0.0,
    live=0,
    model=selected_model,
)
display(test_response)

{'added_obj': None,
 'embedding_down': False,
 'message': '\n\nThe Ask Sage connection is working correctly.',
 'references': '',
 'response': 'OK',
 'status': 200,
 'tool_calls': None,
 'tool_calls_unified': [],
 'tool_responses': [],
 'type': 'completion',
 'usage': None,
 'uuid': '3409c9a5-716e-495a-b54e-9ca86597cf46',
 'vectors_down': False}

## Send Structured Classification Test

In [6]:
structured_client = build_asksage_openai_client(credentials_path=credentials_path)
structured_classifier = AskSageChallengeClassifier(
    model=selected_model,
    credentials_path=credentials_path,
    client=structured_client,
)

test_payload = {
    "instructions": "Choose exactly one predicted_challenge from candidate_challenges.",
    "programming_evidence": {
        "programming_record_id": "connection-test",
        "source_row_number": 0,
        "wbs_name": "Autonomous nuclear enterprise safety analytics",
        "scope_description": "Develop AI tools to improve nuclear enterprise safety analytics and accelerate decision support.",
    },
    "candidate_challenges": [
        {
            "challenge_name": "Nuclear Enterprise Safety Analytics",
            "score": 0.95,
            "matched_terms": ["safety", "analytics", "nuclear enterprise"],
            "has_mstc_context": True,
            "context": {
                "challenge_name": "Nuclear Enterprise Safety Analytics",
                "problem_statements": ["Improve safety analytics and decision support across the nuclear enterprise."],
                "proposed_solutions": ["Apply AI-enabled analytics to safety and operational evidence."],
                "nnsa_advantages": ["Faster evidence-based decisions."],
                "expected_outcomes": ["Improved safety insight and prioritization."],
            },
        },
        {
            "challenge_name": "Manufacturing Process Optimization",
            "score": 0.4,
            "matched_terms": ["optimization"],
            "has_mstc_context": True,
            "context": {
                "challenge_name": "Manufacturing Process Optimization",
                "problem_statements": ["Optimize manufacturing processes."],
                "proposed_solutions": ["Apply AI to production scheduling."],
                "nnsa_advantages": ["Improved throughput."],
                "expected_outcomes": ["Lower production cycle time."],
            },
        },
    ],
}

structured_decision = structured_classifier(test_payload)
display(structured_decision)

{'predicted_challenge': 'Nuclear Enterprise Safety Analytics',
 'confidence': 'high',
 'rationale': 'The scope description explicitly focuses on AI tools for nuclear enterprise safety analytics and decision support, which directly matches the Nuclear Enterprise Safety Analytics challenge’s problem statement and proposed solutions. The WBS name also reinforces this alignment.'}